In [ ]:
!pip install transformers datasets
!pip install tqdm
!pip install peft

In [ ]:
import os
import transformers
import argparse
import torch
import logging
from torch.utils.data import ConcatDataset, DataLoader
from datasets import load_dataset, concatenate_datasets
import re
import sys
import random

import numpy as np
import pickle
import inspect
import torch
import torch.nn as nn

from transformers import (
    T5Config,
    T5ForConditionalGeneration,
    AutoTokenizer,
    LlamaForCausalLM
)

from peft import (  # noqa: E402
    LoraConfig,
    get_peft_model,
    get_peft_model_state_dict,
    prepare_model_for_int8_training,
    set_peft_model_state_dict,
)


In [ ]:
# from https://github.com/agiresearch/OpenP5/blob/main/src/TaskAlternateTrainer.py

from torch.utils.data.distributed import DistributedSampler
from torch.utils.data.sampler import RandomSampler
from torch.utils.data import DataLoader
import inspect
import transformers
from transformers.utils import find_labels

class MultitaskDataloader:
    """
    Data loader that combines and samples from multiple single-task
    data loaders.
    """
    def __init__(self, model, dataset_dict, batch_size, collate_fn, accelerator=None):
        self.model = model
        self.batch_size = batch_size
        self.collate_fn = collate_fn
        self._signature_columns = None

        for task in dataset_dict:
            dataset_dict[task] = self._remove_unused_columns(dataset_dict[task])

        self.dataloader_dict = dict()
        for task in dataset_dict:
            self.dataloader_dict[task] = DataLoader(dataset_dict[task], batch_size = self.batch_size, sampler = RandomSampler(dataset_dict[task]), collate_fn = self.collate_fn)
        if accelerator is not None:
            for task in self.dataloader_dict:
                self.dataloader_dict[task] = accelerator.prepare(self.dataloader_dict[task])

        self.num_batches_dict = {task: len(dataloader) for task, dataloader in self.dataloader_dict.items()}
        self.tasks_list = list(self.dataloader_dict.keys())
        self.num_tasks = len(self.tasks_list)

        self.task_max_num_batch = max(list(self.num_batches_dict.values()))

        self._init()


    def _init(self):
        self.dataloader_iters = [iter(dataloader) for task, dataloader in self.dataloader_dict.items()]
        self.batch_idx = 0


    def _set_signature_columns_if_needed(self):
        if self._signature_columns is None:
            # Inspect model forward signature to keep only the arguments it accepts.
            signature = inspect.signature(self.model.forward)
            self._signature_columns = list(signature.parameters.keys())
            # Labels may be named label or label_ids, the default data collator handles that.
            self._signature_columns += list(set(["label", "label_ids"] + find_labels(self.model.__class__)))

    def _remove_unused_columns(self, dataset):
        self._set_signature_columns_if_needed()
        signature_columns = self._signature_columns

        ignored_columns = list(set(dataset.column_names) - set(signature_columns))
        return dataset.remove_columns(ignored_columns)

    def __len__(self):
        return self.num_tasks * self.task_max_num_batch

    def __iter__(self):
        return self

    def __next__(self):
        task_id = self.batch_idx % self.num_tasks
        self.batch_idx += 1
        # if self.batch_idx < self.num_tasks * self.task_max_num_batch:
        try:
            return next(self.dataloader_iters[task_id])
        except StopIteration:
            self.dataloader_iters[task_id] = iter(self.dataloader_dict[self.tasks_list[task_id]])
            return next(self.dataloader_iters[task_id])
        # self._init()


class TaskAlternateTrainer(transformers.Trainer):

    def get_train_dataloader(self):
        train_dataset = self.train_dataset
        data_collator = self.data_collator

        return MultitaskDataloader(self.model, train_dataset, self._train_batch_size, data_collator, self.accelerator)

In [ ]:
# from https://github.com/agiresearch/OpenP5/blob/main/src/utils/utils.py

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.enabled = False



def random_initialization(model, tokenizer, backbone):
    ids = []
    for x in range(30000):
        tokenized_ids = tokenizer.encode(str(x))
        if 3 in tokenized_ids:
            tokenized_ids.remove(3)
        if 1 in tokenized_ids:
            tokenized_ids.remove(1)
        ids += tokenized_ids
    ids = list(set(ids))

    # reinitialize the embedding in the backbone models
    for index in ids:
        if 't5' in backbone:
            model.shared.weight.data[index] = nn.init.normal_(
                model.shared.weight.data[index], 0, 1.0
            )
        elif 'llama' in backbone.lower():
            model.model.embed_tokens.weight.data[index] = nn.init.normal_(
                model.model.embed_tokens.weight.data[index], 0, 1.0
            )

    return model

def setup_logging(
    log_name: str,
    datasets: str,
    log_dir: str,
    model_dir: str,
    checkpoint_dir: str,
    logging_level: int,
    sample_prompt: str,
    his_prefix: str,
    skip_empty_his: int,
    max_his: int,
    master_port: int,
    tasks: str,
    backbone: str,
    item_indexing: str,
    lr: float,
    epochs: int,
    batch_size: int,
    sample_num: int,
    prompt_file: str
):
    log_name = log_name(
        datasets,
        sample_prompt,
        his_prefix,
        skip_empty_his,
        max_his,
        master_port,
        tasks,
        backbone,
        item_indexing,
        lr,
        epochs,
        batch_size,
        sample_num,
        prompt_file
    )
    if len(datasets.split(',')) > 1:
        folder_name = 'SP5'
    else:
        folder_name = datasets
    folder = os.path.join(log_dir, folder_name)
    if not os.path.exists(folder):
        os.makedirs(folder)
    log_file = os.path.join(log_dir, folder_name, log_name + '.log')

    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)
    logging.basicConfig(filename=log_file, level=logging_level, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    logging.getLogger().addHandler(logging.StreamHandler(sys.stdout))

    return

def log_name(
    datasets,
    sample_prompt,
    his_prefix,
    skip_empty_his,
    max_his,
    master_port,
    tasks,
    backbone,
    item_indexing,
    lr,
    epochs,
    batch_size,
    sample_num,
    prompt_file,
):
    if len(datasets.split(',')) > 1:
        folder_name = 'SP5'
    else:
        folder_name = datasets
    params = [str(sample_prompt), str(his_prefix), str(skip_empty_his), str(max_his), str(master_port), folder_name, tasks, backbone, item_indexing, str(lr), str(epochs), str(batch_size), sample_num, prompt_file[3:-4]]
    return '_'.join(params)

def ReadLineFromFile(path):
    if not os.path.exists(path):
        raise FileNotFoundError
    lines = []
    with open(path,'r') as fd:
        for line in fd:
            lines.append(line.rstrip('\n'))
    return lines

def WriteDictToFile(path, write_dict):
    with open(path, 'w') as out:
        for user, items in write_dict.items():
            if type(items) == list:
                out.write(user + ' ' + ' '.join(items) + '\n')
            else:
                out.write(user + ' ' + str(items) + '\n')


def load_prompt_template(path, task_list):
    """
    Load prompt template from the file. Keep training tasks only.
    Input:
    - path: The path for prompt template txt file.
    - task_list: A list of required tasks.
    Return:
    - prompt_templates: a dictionary of prompt templates. e.g., {task: {'seen': {'0': {'Input': template_input, 'Output': template_output}}}}

    """

    if not os.path.exists(path):
        raise FileNotFoundError
    prompt_info = ReadLineFromFile(path)
    prompt_templates = dict()
    for prompt in prompt_info:
        t = [sens.strip() for sens in prompt.split(';')]
        if t[0] not in task_list:
            continue
        if t[0] not in prompt_templates:
            prompt_templates[t[0]] = dict()
        if t[1] not in prompt_templates[t[0]]:
            prompt_templates[t[0]][t[1]] = dict()
        num = len(prompt_templates[t[0]][t[1]])
        prompt_templates[t[0]][t[1]][str(num)] = dict()
        prompt_templates[t[0]][t[1]][str(num)]['Input'] = t[2]
        prompt_templates[t[0]][t[1]][str(num)]['Output'] = t[3]
    return prompt_templates

def get_info_from_prompt(prompt_templates):
    """
    Extract the require information from the prompt templates.
    Input:
    - prompt_templates: a dictionary of prompt templates.
    Output:
    - info: a list of required information.
    """

    info = []
    for task in prompt_templates:
        for see in prompt_templates[task]:
            for i in prompt_templates[task][see]:
                info += re.findall(r'\{.*?\}', prompt_templates[task][see][i]['Input'])
                info += re.findall(r'\{.*?\}', prompt_templates[task][see][i]['Output'])
    info = [i[1:-1] for i in set(info)]
    return info

def check_task_prompt(prompt_templates, task_list):
    """
    Check if all tasks have prompt templates. Raise Error if training tasks have no prompt.
    Input:
    - prompt_templates: A dictionary of prompt templates.
    - task_list: A list of training tasks.
    """
    for task in task_list:
        assert task in prompt_templates, f"No prompt for {task} task"

In [ ]:
# from https://github.com/agiresearch/OpenP5/blob/main/src/utils/indexing.py

def sequential_indexing(data_path, dataset, user_sequence_dict, order):
    """
    Use sequential indexing method to index the given user seuqnece dict.
    """
    user_index_file = os.path.join(data_path, dataset, 'user_indexing.txt')
    item_index_file = os.path.join(data_path, dataset, f'item_sequential_indexing_{order}.txt')
    reindex_sequence_file = os.path.join(data_path, dataset, f'user_sequence_sequential_indexing_{order}.txt')

    if os.path.exists(reindex_sequence_file):
        user_sequence = ReadLineFromFile(reindex_sequence_file)

        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)

        return construct_user_sequence_dict(user_sequence), item_map

    # For user index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(user_index_file):
        user_info = ReadLineFromFile(user_index_file)
        user_map = get_dict_from_lines(user_info)
    else:
        user_map = generate_user_map(user_sequence_dict)
        WriteDictToFile(user_index_file, user_map)


    # For item index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(item_index_file):
        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)
    else:
        item_map = dict()
        if order == 'original':
            user_list = user_sequence_dict.keys()
        elif order == 'short2long':
            user_list = sorted(user_sequence_dict, key=lambda x: len(user_sequence_dict[x]), reverse=False)
        elif order == 'long2short':
            user_list = sorted(user_sequence_dict, key=lambda x: len(user_sequence_dict[x]), reverse=True)

        for user in user_list:
            items = user_sequence_dict[user][:-2]
            for item in items:
                if item not in item_map:
                    item_map[item] = str(len(item_map) + 1001)
        for user in user_list:
            items = user_sequence_dict[user][-2:]
            for item in items:
                if item not in item_map:
                    item_map[item] = str(len(item_map) + 1001)
        WriteDictToFile(item_index_file, item_map)

    reindex_user_sequence_dict = reindex(user_sequence_dict, user_map, item_map)
    WriteDictToFile(reindex_sequence_file, reindex_user_sequence_dict)
    return reindex_user_sequence_dict, item_map



def random_indexing(data_path, dataset, user_sequence_dict):
    """
    Use random indexing method to index the given user seuqnece dict.
    """
    user_index_file = os.path.join(data_path, dataset, 'user_indexing.txt')
    item_index_file = os.path.join(data_path, dataset, 'item_random_indexing.txt')
    reindex_sequence_file = os.path.join(data_path, dataset, f'user_sequence_random_indexing.txt')

    if os.path.exists(reindex_sequence_file):
        user_sequence = ReadLineFromFile(reindex_sequence_file)

        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)

        return construct_user_sequence_dict(user_sequence), item_map

    # For user index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(user_index_file):
        user_info = ReadLineFromFile(user_index_file)
        user_map = get_dict_from_lines(user_info)
    else:
        user_map = generate_user_map(user_sequence_dict)
        WriteDictToFile(user_index_file, user_map)


    # For item index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(item_index_file):
        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)
    else:
        item_map = dict()
        items = set()
        for user in user_sequence_dict:
            items.update(user_sequence_dict[user])
        items = list(items)
        random.shuffle(items)
        for item in items:
            if item not in item_map:
                item_map[item] = str(len(item_map) + 1001)
        WriteDictToFile(item_index_file, item_map)

    reindex_user_sequence_dict = reindex(user_sequence_dict, user_map, item_map)
    WriteDictToFile(reindex_sequence_file, reindex_user_sequence_dict)
    return reindex_user_sequence_dict, item_map

def collaborative_indexing(data_path, dataset, user_sequence_dict, token_size, cluster_num, last_token, float32):
    """
    Use collaborative indexing method to index the given user seuqnece dict.
    """
    user_index_file = os.path.join(data_path, dataset, 'user_indexing.txt')
    item_index_file = os.path.join(data_path, dataset, f'item_collaborative_indexing_{token_size}_{cluster_num}_{last_token}.txt')
    reindex_sequence_file = os.path.join(data_path, dataset, f'user_sequence_collaborative_indexing_{token_size}_{cluster_num}_{last_token}.txt')

    if os.path.exists(reindex_sequence_file):
        user_sequence = ReadLineFromFile(reindex_sequence_file)

        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)

        return construct_user_sequence_dict(user_sequence), item_map

    # For user index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(user_index_file):
        user_info = ReadLineFromFile(user_index_file)
        user_map = get_dict_from_lines(user_info)
    else:
        user_map = generate_user_map(user_sequence_dict)
        WriteDictToFile(user_index_file, user_map)


    # For item index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(item_index_file):
        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)
    else:
        item_map = generate_collaborative_id(user_sequence_dict, token_size, cluster_num, last_token, float32)
        WriteDictToFile(item_index_file, item_map)

    reindex_user_sequence_dict = reindex(user_sequence_dict, user_map, item_map)
    WriteDictToFile(reindex_sequence_file, reindex_user_sequence_dict)
    return reindex_user_sequence_dict, item_map

def generate_collaborative_id(user_sequence_dict, token_size, cluster_num, last_token, float32):
    """
    Generate collaborative index for items.
    """
    # get the items in training data and all data.
    all_items = set()
    train_items = set()
    for user in user_sequence_dict:
        all_items.update(set(user_sequence_dict[user]))
        train_items.update(set(user_sequence_dict[user][:-2]))

    # reindex all training items for calculating the adjacency matrix
    item2id = dict()
    id2item = dict()
    for item in train_items:
        item2id[item] = len(item2id)
        id2item[len(id2item)] = item


    # calculate the co-occurrence of items in the training data as an adjacency matrix
    if float32 > 0:
        adj_matrix = np.zeros((len(item2id), len(item2id)), dtype=np.float32)
    else:
        adj_matrix = np.zeros((len(item2id), len(item2id)))
    for user in user_sequence_dict:
        interactions = user_sequence_dict[user][:-2]
        for pairs in combinations(interactions, 2):
            adj_matrix[item2id[pairs[0]]][item2id[pairs[1]]] += 1
            adj_matrix[item2id[pairs[1]]][item2id[pairs[0]]] += 1


    # get the clustering results for the first layer
    clustering = SpectralClustering(
        n_clusters=cluster_num,
        assign_labels="cluster_qr",
        random_state=0,
        affinity="precomputed",
    ).fit(adj_matrix)
    labels = clustering.labels_.tolist()

    # count the clustering results
    grouping = defaultdict(list)
    for i in range(len(labels)):
        grouping[labels[i]].append((id2item[i],i))

    item_map = dict()
    index_now = 0

    # add current clustering information into the item indexing results.
    item_map, index_now = add_token_to_indexing(item_map, grouping, index_now, token_size)

    # add current clustering info into a queue for BFS
    queue = []
    for group in grouping:
        queue.append(grouping[group])

    # apply BFS to further use spectral clustering for large groups (> token_size)
    while queue:
        group_items = queue.pop(0)

        # if current group is small enough, add the last token to item indexing
        if len(group_items) <= token_size:
            item_list = [items[0] for items in group_items]
            if last_token == 'sequential':
                item_map = add_last_token_to_indexing_sequential(item_map, item_list, token_size)
            elif last_token == 'random':
                item_map = add_last_token_to_indexing_random(item_map, item_list, token_size)
        else:
            # calculate the adjacency matrix for current group
            if float32 > 0:
                sub_adj_matrix = np.zeros((len(group_items), len(group_items)), dtype=np.float32)
            else:
                sub_adj_matrix = np.zeros((len(group_items), len(group_items)))
            for i in range(len(group_items)):
                for j in range(i+1, len(group_items)):
                    sub_adj_matrix[i][j] = adj_matrix[group_items[i][1]][group_items[j][1]]
                    sub_adj_matrix[j][i] = adj_matrix[group_items[j][1]][group_items[i][1]]

            # get the clustering results for current group
            clustering = SpectralClustering(
                n_clusters=cluster_num,
                assign_labels="cluster_qr",
                random_state=0,
                affinity="precomputed",
            ).fit(sub_adj_matrix)
            labels = clustering.labels_.tolist()

            # count current clustering results
            grouping = defaultdict(list)
            for i in range(len(labels)):
                grouping[labels[i]].append(group_items[i])

            # add current clustering information into the item indexing results.
            item_map, index_now = add_token_to_indexing(item_map, grouping, index_now, token_size)

            # push current clustering info into the queue
            for group in grouping:
                queue.append(grouping[group])

    # if some items are not in the training data, assign an index for them
    remaining_items = list(all_items - train_items)
    if len(remaining_items) > 0:
        if last_token == 'sequential':
            item_map = add_last_token_to_indexing_sequential(item_map, remaining_items, token_size)
        elif last_token == 'random':
            item_map = add_last_token_to_indexing_random(item_map, remaining_items, token_size)

    return item_map



def add_token_to_indexing(item_map, grouping, index_now, token_size):
    for group in grouping:
        index_now = index_now % token_size
        for (item, idx) in grouping[group]:
            if item not in item_map:
                item_map[item] = ''
            item_map[item] += f'<CI{index_now}>'
        index_now += 1
    return item_map, index_now

def add_last_token_to_indexing_random(item_map, item_list, token_size):
    last_tokens = random.sample([i for i in range(token_size)], len(item_list))
    for i in range(len(item_list)):
        item = item_list[i]
        if item not in item_map:
            item_map[item] = ''
        item_map[item] += f'<CI{last_tokens[i]}>'
    return item_map

def add_last_token_to_indexing_sequential(item_map, item_list, token_size):
    for i in range(len(item_list)):
        item = item_list[i]
        if item not in item_map:
            item_map[item] = ''
        item_map[item] += f'<CI{i}>'
    return item_map


def get_dict_from_lines(lines):
    """
    Used to get user or item map from lines loaded from txt file.
    """
    index_map = dict()
    for line in lines:
        info = line.split(" ")
        index_map[info[0]] = info[1]
    return index_map




def generate_user_map(user_sequence_dict):
    """
    generate user map based on user sequence dict.
    """
    user_map = dict()
    for user in user_sequence_dict.keys():
        user_map[user] = str(len(user_map) + 1)
    return user_map


def reindex(user_sequence_dict, user_map, item_map):
    """
    reindex the given user sequence dict by given user map and item map
    """
    reindex_user_sequence_dict = dict()
    for user in user_sequence_dict:
        uid = user_map[user]
        items = user_sequence_dict[user]
        reindex_user_sequence_dict[uid] = [item_map[i] for i in items]

    return reindex_user_sequence_dict


def construct_user_sequence_dict(user_sequence):
    """
    Convert a list of string to a user sequence dict. user as key, item list as value.
    """

    user_seq_dict = dict()
    for line in user_sequence:
        user_seq = line.split(" ")
        user_seq_dict[user_seq[0]] = user_seq[1:]
    return user_seq_dict

In [ ]:
# from https://github.com/agiresearch/OpenP5/blob/main/src/train.py

def train(
    seed: int = 2023,
    model_dir: str = "../model",
    checkpoint_dir: str = "../checkpoint",
    model_name: str = "model.pt",
    log_dir: str = "../log",
    master_addr: str = "localhost",
    master_port: str = "12345",
    logging_level: int = logging.INFO,
    data_path: str = "../data",
    item_indexing: str = "sequential",
    tasks: str = 'sequential,straightforward',
    datasets_: str = "Beauty",
    prompt_file: str = '../prompt.txt',
    sequential_orderl: str = "original",
    collaborative_token_size: int = 200,
    collaborative_cluster: int = 20,
    collaborative_last_token: str = "sequential",
    collaborative_float32: int = 0,
    max_his: int = 10,
    his_prefix: int = 1,
    his_sep:str = " , ",
    skip_empty_his: str = 1,
    valid_prompt: str = 'seen:0',
    valid_prompt_sample: int = 1,
    valid_sample_num: str = '3,3',
    test_prompt: str = 'seen:0',
    sample_prompt: int = 0,
    sample_num: str = '2,2',
    cutoff: int = 1024,
    batch_size: int = 32,
    eval_batch_size: int = 32,
    group_task_in_batch: int = 1,
    task_alternating_optim: int = 0,
    optim: str = "adamw_torch",
    epochs: int = 10,
    lr: float = 1e-3,
    clip: float = 1,
    logging_steps: int= 10,
    warmup_steps: int = 100,
    gradient_accumulation_steps: int = 16,
    weight_decay: float = 0.01,
    adam_eps: float = 1e-6,
    dropout: float = 0.1,
    alpha: float = 2,
    backbone: str = 't5-small',
    random_initialize: int = 1,
    test_epoch: int = 1,
    valid_select: int = 0,
    lora: int = 0,
    lora_r: int = 8,
    lora_alpha: int = 16,
    lora_dropout: float = 0.05,
    lora_target_modules: str = 'q_proj,v_proj,embed_tokens',
    metrics: str = 'hit@5,hit@10,ndcg@5,ndcg@10'
):

    # setup
    setup_logging(
        log_name,
        datasets_,
        log_dir,
        model_dir,
        checkpoint_dir,
        logging_level,
        sample_prompt,
        his_prefix,
        skip_empty_his,
        max_his,
        master_port,
        tasks,
        backbone,
        item_indexing,
        lr,
        epochs,
        batch_size,
        sample_num,
        prompt_file
    )

    set_seed(seed)

    # determine whether distributed
    device_map = "auto"
    world_size = int(os.environ.get("WORLD_SIZE", 1))
    ddp = world_size != 1
    if ddp:
        device_map = {"": int(os.environ.get("LOCAL_RANK") or 0)}
        gradient_accumulation_steps = gradient_accumulation_steps // world_size

    # use wandb
    wandb_project = ""
    wandb_run_name = ""
    wandb_watch = ""  # options: false | gradients | all
    wandb_log_model = ""
    use_wandb = len(wandb_project) > 0 or (
        "WANDB_PROJECT" in os.environ and len(os.environ["WANDB_PROJECT"]) > 0
    )
    # Only overwrite environ if wandb param passed
    if len(wandb_project) > 0:
        os.environ["WANDB_PROJECT"] = wandb_project
    if len(wandb_watch) > 0:
        os.environ["WANDB_WATCH"] = wandb_watch
    if len(wandb_log_model) > 0:
        os.environ["WANDB_LOG_MODEL"] = wandb_log_model

    # load model, tokenizer
    if 't5' in backbone.lower():
        config = T5Config.from_pretrained(backbone)
        model = T5ForConditionalGeneration.from_pretrained(backbone, config=config)
        tokenizer = AutoTokenizer.from_pretrained(backbone)
    elif 'llama' in backbone.lower():
        model = LlamaForCausalLM.from_pretrained(
            'meta-llama/' + backbone,
            load_in_8bit=True,
            torch_dtype=torch.float16
        )
        tokenizer = AutoTokenizer.from_pretrained('meta-llama/' + backbone)
    else:
        raise NotImplementError




    datasets = datasets_.split(',')
    if len(datasets) == 1:
        dataset = datasets[0]
        train_data_file = os.path.join(data_path, dataset, f'{dataset}_{tasks}_{item_indexing}_train.json')
        valid_data_file = os.path.join(data_path, dataset, f'{dataset}_{tasks}_{item_indexing}_validation_{valid_prompt}.json')
        train_data = load_dataset("json", data_files=train_data_file, field='data')
        valid_data = load_dataset("json", data_files=valid_data_file, field='data')
    else:
        train_data_list, valid_data_list = [], []
        for dataset in datasets:
            train_data_file = os.path.join(data_path, dataset, f'{dataset}_{tasks}_{item_indexing}_train.json')
            valid_data_file = os.path.join(data_path, dataset, f'{dataset}_{tasks}_{item_indexing}_validation_{valid_prompt}.json')
            t_data = load_dataset("json", data_files=train_data_file, field='data')
            v_data = load_dataset("json", data_files=valid_data_file, field='data')
            train_data_list.append(t_data)
            valid_data_list.append(v_data)
        train_data = concatenate_datasets(train_data_list)
        valid_data = concatenate_datasets(valid_data_list)

    def tokenize(prompt, add_eos_token=True):
        # there's probably a way to do this with the tokenizer settings
        # but again, gotta move fast
        result = tokenizer(
            prompt, truncation=True, max_length=cutoff, padding=False, return_tensors=None,
        )
        if (isinstance(result["input_ids"][-1], int) and result["input_ids"][-1] != tokenizer.eos_token_id
            and len(result["input_ids"]) < cutoff
            and add_eos_token
           ):
            result["input_ids"].append(tokenizer.eos_token_id)
            result["attention_mask"].append(1)
        elif isinstance(result["input_ids"][-1], list) and add_eos_token:
            for i in range(len(result['input_ids'])):
                if result["input_ids"][i][-1] != tokenizer.eos_token_id and len(result["input_ids"][i]) < cutoff:
                    result["input_ids"][i].append(tokenizer.eos_token_id)
                    result["attention_mask"][i].append(1)

        result["labels"] = result["input_ids"].copy()

        return result

    def generate_prompt(data_point):
        return f'{data_point["input"]} {data_point["output"]}'

    def process_func(datapoint):
        if 't5' in backbone.lower():
            encoding = tokenize(datapoint['input'], add_eos_token=True)
            labels = tokenize(datapoint['output'], add_eos_token=True)
            encoding['labels'] = labels['input_ids'].copy()
        elif 'llama' in backbone.lower():
            user_prompt = generate_prompt({**datapoint, "output": ""})
            encoding_input = tokenize(user_prompt, add_eos_token=False)
            input_len = len(encoding_input["input_ids"])
            full_prompt = generate_prompt(datapoint)
            encoding = tokenize(full_prompt)

            encoding["labels"] = (
                [-100] * input_len
                + encoding["labels"][input_len:]
            )

        # return encoding
        return {**datapoint, **encoding}

    # add token and resize embedding for collaborative indexing
    if item_indexing == 'collaborative':
        new_tokens = []
        for dataset in datasets:
            item_index_file = os.path.join(data_path, dataset, f'item_collaborative_indexing_{collaborative_token_size}_{collaborative_cluster}_{collaborative_last_token}.txt')
            item_info = ReadLineFromFile(item_index_file)
            item_map = get_dict_from_lines(item_info)
            for idx in list(item_map.values()):
                new_token += re.findall(r'\<.*?\>', idx)
        tokenizer.add_tokens(ds.new_token)
        model.resize_token_embeddings(len(tokenizer))

    # no task alternating optimization if only one task in the data
    if len(set(train_data['train']['task'])) == 1:
        task_alternating_optim = 0

    if task_alternating_optim == 1:
        TrainSet = dict()
        for task in set(train_data['train']['task']):
            TrainSet[task] = train_data['train'].filter(lambda example: example["task"]==task)
        for task in TrainSet:
            TrainSet[task] = TrainSet[task].shuffle().map(process_func, batched=True)

    else:
        TrainSet = train_data['train'].shuffle().map(process_func, batched=True)

    ValidSet = valid_data['train'].shuffle().map(process_func, batched=True)



    # randomly initialize number related tokens
    if random_initialize == 1:
        # logging.info("Random initialize number related tokens")
        random_initialization(model, tokenizer, backbone)

    # apply lora
    if lora > 0:
        model = prepare_model_for_int8_training(model)

        config = LoraConfig(
                r=lora_r,
                lora_alpha=lora_alpha,
                target_modules=lora_target_modules.split(','),
                lora_dropout=lora_dropout,
                bias="none",
                task_type="CAUSAL_LM",
            )
        model = get_peft_model(model, config)
        model.print_trainable_parameters()

    tokenizer.pad_token_id = (
        0  # unk. we want this to be different from the eos token
    )
    tokenizer.padding_side = "left"

    # decide output dir
    if len(datasets_.split(',')) > 1:
        folder_name = 'SP5'
    else:
        folder_name = datasets_
    output_dir = os.path.join(model_dir, folder_name, item_indexing, backbone)

    if task_alternating_optim == 1:
        trainer = TaskAlternateTrainer(model=model,
            train_dataset=TrainSet,
            eval_dataset=ValidSet if valid_select > 0 else None,
            args= transformers.TrainingArguments(
                per_device_train_batch_size=batch_size,
                gradient_accumulation_steps=gradient_accumulation_steps,
                warmup_steps=warmup_steps,
                num_train_epochs=epochs,
                learning_rate=lr,
                fp16=True,
                logging_dir=log_dir,
                logging_steps=logging_steps,
                optim=optim,
                evaluation_strategy="steps" if valid_select > 0 else "no",
                save_strategy="steps",
                eval_steps=200 if valid_select > 0 else None,
                save_steps=200,
                output_dir=output_dir,
                save_total_limit=3,
                load_best_model_at_end=True if valid_select > 0 else False,
                ddp_find_unused_parameters=False if ddp else None,
                group_by_length=False,
                report_to="wandb" if use_wandb else None,
                run_name=wandb_run_name if use_wandb else None,
            ),
            data_collator=transformers.DataCollatorForSeq2Seq(
                tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
            ),
        )
    else:
        trainer = transformers.Trainer(
            model=model,
            train_dataset=TrainSet,
            eval_dataset=ValidSet if valid_select > 0 else None,
            args= transformers.TrainingArguments(
                per_device_train_batch_size=batch_size,
                gradient_accumulation_steps=gradient_accumulation_steps,
                warmup_steps=warmup_steps,
                num_train_epochs=epochs,
                learning_rate=lr,
                fp16=True,
                logging_steps=logging_steps,
                optim=optim,
                evaluation_strategy="steps" if valid_select > 0 else "no",
                save_strategy="steps",
                eval_steps=200 if valid_select > 0 else None,
                save_steps=200,
                output_dir=output_dir,
                save_total_limit=3,
                load_best_model_at_end=True if valid_select > 0 else False,
                ddp_find_unused_parameters=False if ddp else None,
                group_by_length=False,
                report_to="wandb" if use_wandb else None,
                run_name=wandb_run_name if use_wandb else None,
            ),
            data_collator=transformers.DataCollatorForSeq2Seq(
                tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
            ),
        )

    trainer.train()

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

In [ ]:
train(
    model_dir="/content/drive/MyDrive/OpenP5/models/",
    log_dir="/content/drive/MyDrive/OpenP5/logs/",
    master_port=1234,
    item_indexing="sequential",
    tasks="sequential,straightforward",
    datasets_="Beauty",
    data_path = '/content/drive/MyDrive/OpenP5/data/',
    epochs=10,
    batch_size=512,
    backbone="t5-small",
    cutoff=1024
)